# Silver Layer – ERP Customer Information

This notebook transforms ERP customer data from the Bronze layer into the Silver layer by applying data cleaning, validation, standardization, and transformation rules.

In [0]:
from pyspark.sql.functions import *
df = spark.read.table('workspace.bronze.erp_cust_az12')

## Prepare Customer ID for Join

Remove the first three characters from the customer ID to match the key format in the related table.

In [0]:
df = ( df.withColumn('CID' ,  
                  substring(col("CID"),4,length(col("CID")) )) 
    
)

## Clean Birth Dates

Replace invalid future birth dates with null values.

In [0]:
df = (df.withColumn('BDATE' ,
               when(col("BDATE") > current_date() , None)
 .otherwise(col('BDATE'))  )
)

## Standardize Gender

Convert different gender representations (e.g., M/Male, F/Female) into a consistent format.

In [0]:
df = (df.withColumn('GEN' ,   
               when( trim(col("GEN")).isin('Male' ,'M') , 'Male')
                .when(trim(col("GEN")).isin('Female' , 'F') , 'Female')
                .otherwise('N/A')
                )
)



## Rename Columns

Rename columns to follow the Silver layer naming convention.

In [0]:
rename_map = {
    'CID' : 'customer_number' , 
    'BDATE' : 'birth_date' ,
    'GEN' : 'gender'
}
for old_name , new_name in rename_map.items() :
   df =  df.withColumnRenamed(old_name,new_name)

## Load to Silver Layer

Write the cleaned and transformed customer data to the Silver layer.

In [0]:
df.write.mode('overwrite').saveAsTable('workspace.silver.erp_customers')

## Verify Data Load

Validate that the customer data has been successfully loaded into the Silver layer.

In [0]:
%sql
SELECT *
FROM workspace.silver.erp_customers LIMIT 10